# Практика: Python

Укажите **ФИО**, **группу** и **кредит** → нажмите **Сгенерировать задание** → напишите решение в ячейке ниже → нажмите **Проверить**.

In [ ]:
# Ключи автоматически подгружаются с GitHub (см. GITHUB_SECRETS_URL ниже).
# Преподаватель: создайте секретный Gist или файл в репо с JSON и укажите raw-URL.
GITHUB_SECRETS_URL = "https://gist.githubusercontent.com/argokz/ba6ddd31a31fd8502f0b5eb46d9bba36/raw/6ab1ed1e2689c186bbd3697ca2461bb858c60e1c/gistfile_1.json"
GITHUB_TOKEN = ""        # для private repo (опционально)
OPENAI_API_KEY = ""
DB_PASSWORD = ""

!pip install -q openai psycopg2-binary ipywidgets requests

import os
import json
import hashlib
import urllib.request
from openai import OpenAI
import ipywidgets as widgets
from IPython.display import display, clear_output

# Автоматическое чтение ключей с GitHub (Gist или raw-файл репо)
if GITHUB_SECRETS_URL and GITHUB_SECRETS_URL.strip():
    try:
        req = urllib.request.Request(GITHUB_SECRETS_URL.strip())
        if GITHUB_TOKEN and GITHUB_TOKEN.strip():
            req.add_header("Authorization", f"token {GITHUB_TOKEN.strip()}")
        with urllib.request.urlopen(req, timeout=10) as resp:
            data = json.loads(resp.read().decode())
            OPENAI_API_KEY = data.get("OPENAI_API_KEY", OPENAI_API_KEY) or ""
            DB_PASSWORD = data.get("DB_PASSWORD", DB_PASSWORD) or ""
    except Exception:
        pass

API_KEY = (OPENAI_API_KEY or os.environ.get("OPENAI_API_KEY") or "").strip()
if not API_KEY:
    try:
        from google.colab import userdata
        API_KEY = userdata.get("OPENAI_API_KEY", "").strip()
    except Exception:
        pass
if not API_KEY:
    API_KEY = input("OpenAI API ключ: ").strip()

client = OpenAI(api_key=API_KEY)
MODEL = "gpt-4o-mini"

db_password = (DB_PASSWORD or os.environ.get("DB_PASSWORD") or "").strip()
if not db_password:
    try:
        from google.colab import userdata
        db_password = userdata.get("DB_PASSWORD", "").strip()
    except Exception:
        pass
if not db_password:
    db_password = input("Пароль БД: ").strip()

DB = {
    "host": os.environ.get("DB_HOST", "145.249.247.138"),
    "port": int(os.environ.get("DB_PORT", "5440")),
    "dbname": os.environ.get("DB_NAME", "exam_kmu"),
    "user": os.environ.get("DB_USER", "postgres"),
    "password": db_password,
}

print("Готово.")

In [ ]:
def generate_assignment(variant: int) -> str:
    prompt = f"""
Сгенерируй ОДНО задание по программированию на Python для варианта {variant}.
Требования: ввод/вывод (input/print), условные операторы (if/elif/else), сравнение строк, циклы (for/while), строки, списки, кортежи. Один связный сценарий, 15-25 строк кода. Язык: русский. Ответ — только текст задания, без кода.
"""
    r = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0.7)
    return r.choices[0].message.content.strip()

def evaluate_solution(assignment_text: str, code: str, variant: int) -> dict:
    prompt = f"""
Задание (вариант {variant}):\n{assignment_text}\n\nКод студента:\n```python\n{code}\n```
Проверь решение. Ответь СТРОГО JSON: {{\"score\": <0-100>, \"feedback\": \"комментарий на русском\"}}
"""
    r = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": prompt}], temperature=0.3)
    text = r.choices[0].message.content.strip()
    if text.startswith("```"):
        text = text.split("```")[1].lstrip("json\n")
    try:
        d = json.loads(text)
        score = max(0, min(100, int(d.get("score", 0))))
        return {"score": score, "feedback": d.get("feedback", "")}
    except Exception:
        return {"score": 0, "feedback": "Ошибка разбора ответа."}

def save_to_db(fio: str, group_name: str, credit: str, variant: int, assignment: str, code: str, score: int, feedback: str):
    import psycopg2
    conn = psycopg2.connect(**DB)
    cur = conn.cursor()
    cur.execute(
        """INSERT INTO results (fio, group_name, credit, variant, assignment, code, score, feedback)
           VALUES (%s, %s, %s, %s, %s, %s, %s, %s)""",
        (fio, group_name, credit, variant, assignment[:5000] if assignment else None, code[:10000] if code else None, score, feedback[:2000] if feedback else None)
    )
    conn.commit()
    cur.close()
    conn.close()

# Глобальное состояние для кнопки «Проверить»
current_assignment = ""
current_variant = 1
current_fio = ""
current_group = ""
current_credit = ""

In [ ]:
fio_in = widgets.Text(placeholder="ФИО", description="ФИО:", style={"description_width": "60px"})
group_in = widgets.Text(placeholder="Группа", description="Группа:", style={"description_width": "60px"})
credit_in = widgets.Text(placeholder="Кредит", description="Кредит:", style={"description_width": "60px"})
out_assign = widgets.Output()

def on_generate(b):
    global current_assignment, current_variant, current_fio, current_group, current_credit
    fio, group, credit = fio_in.value.strip(), group_in.value.strip(), credit_in.value.strip()
    if not fio or not group or not credit:
        with out_assign:
            clear_output(wait=True)
            print("Заполните ФИО, группу и кредит.")
        return
    h = int(hashlib.sha256(f"{credit}{group}{fio}".encode()).hexdigest(), 16) % 17
    current_variant = h + 1
    current_fio, current_group, current_credit = fio, group, credit
    with out_assign:
        clear_output(wait=True)
        print("Генерация задания...")
    current_assignment = generate_assignment(current_variant)
    with out_assign:
        clear_output(wait=True)
        print(f"Вариант {current_variant}")
        print("-" * 40)
        print(current_assignment)

btn_gen = widgets.Button(description="Сгенерировать задание", button_style="primary")
btn_gen.on_click(on_generate)

display(widgets.VBox([fio_in, group_in, credit_in, btn_gen, out_assign]))

Напишите решение в ячейке ниже (переменная `student_code`) и **запустите ячейку**, затем нажмите **Проверить**.

In [ ]:
student_code = """
# Ваше решение
pass
"""

In [ ]:
out_check = widgets.Output()

def on_check(b):
    global current_assignment, current_variant, current_fio, current_group, current_credit
    with out_check:
        clear_output(wait=True)
        print("Проверка...")
    code = ""
    try:
        code = get_ipython().user_ns.get("student_code", "")
    except Exception:
        code = ""
    if not code or code.strip() in ("", "pass", "# Ваше решение"):
        with out_check:
            clear_output(wait=True)
            print("Сначала напишите решение в ячейке выше и запустите её.")
        return
    if not current_assignment:
        with out_check:
            clear_output(wait=True)
            print("Сначала нажмите «Сгенерировать задание».")
        return
    result = evaluate_solution(current_assignment, code, current_variant)
    try:
        save_to_db(current_fio, current_group, current_credit, current_variant, current_assignment, code, result["score"], result["feedback"])
    except Exception as e:
        pass  # не показываем ошибку БД студенту
    with out_check:
        clear_output(wait=True)
        print(f"Оценка: {result['score']} / 100")
        print(f"Комментарий: {result['feedback']}")

btn_check = widgets.Button(description="Проверить", button_style="success")
btn_check.on_click(on_check)
display(widgets.VBox([btn_check, out_check]))